# 01 Ingest NEMWeb ZIP Files

List enabled AEMO NEMWeb source folders, skip ZIPs already recorded in the manifest, land unseen ZIP files, and append ingestion control records. The same ingestion algorithm runs locally with CSV control files or in Fabric with Lakehouse Files and Delta tables.


## Configure Run Parameters

This cell auto-detects whether the notebook is running locally or in Fabric, then defines ingestion parameters.


In [1]:
# Cell purpose: Configure Run Parameters.
from pathlib import Path
import os
import sys
import uuid

# Runtime mode is detected automatically. Fabric notebooks expose a Spark session.
try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

# Fabric pipeline parameters. Override these in the pipeline activity where required.
run_id = str(uuid.uuid4())
source_name = ""  # Blank means all enabled sources.
max_zips_per_run = 500
lookback_hours = 6
dry_run = False

# Local-only output folder names. Ignored when is_local_run is False.
local_output_folder = "data"
local_raw_root = "files/nemweb/raw_zip"

print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")


run_id=2a49a34e-d84a-4a7a-afb2-6648ffb22aa7
runtime=local


## Resolve Runtime Paths

This cell resolves local or Fabric package paths before importing project modules.


In [2]:
# Cell purpose: Resolve Runtime Paths.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


repo_root = None
local_output_root = None
package_paths = []

if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / local_output_folder
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))
        print("Python search paths added:", str(package_path))

if is_local_run:
    print(f"Local output root: {local_output_root}")


Python search paths added: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\src
Local output root: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data


## Import Project Modules

This cell imports project modules after runtime paths are resolved. Runtime-specific imports are gated so Fabric does not import local-only modules.


In [3]:
# Cell purpose: Import Project Modules.
import importlib

from nem_fabric import common_ingestion
from nem_fabric.common_config import load_yaml
from nem_fabric.common_ingestion import select_enabled_sources

runtime_ingestion = importlib.import_module(
    "nem_fabric.local_ingestion" if is_local_run else "nem_fabric.fabric_ingestion"
)


## Select Runtime Store

This cell creates the storage backend for the selected runtime.


In [4]:
# Cell purpose: Select Runtime Store.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    store = runtime_ingestion.LocalCsvIngestionStore(local_output_root)
else:
    store = runtime_ingestion.FabricSparkIngestionStore(
        spark,
        lakehouse_root="/lakehouse/default",
    )

print(f"Runtime store: {'local' if is_local_run else 'fabric'}")


Runtime store: local


## Load Source Configuration

This cell reads `config/sources.yml`, keeps enabled source folders, and optionally filters to one source. It prevents accidental runs when no source matches.


In [5]:
# Cell purpose: Load Source Configuration.
sources_config = load_yaml("config/sources.yml")
sources = select_enabled_sources(sources_config, source_name=source_name)

print("Sources selected:", [source["name"] for source in sources])


Sources selected: ['DispatchIS_Reports']


## Run Ingestion

This cell runs the shared ingestion algorithm. Storage-specific behaviour is handled by the selected store.


In [6]:
# Cell purpose: Run Ingestion.
ingestion_config = common_ingestion.IngestionConfig(
    run_id=run_id,
    source_name=source_name,
    max_zips_per_run=max_zips_per_run,
    lookback_hours=lookback_hours,
    dry_run=dry_run,
    raw_root=local_raw_root if is_local_run else common_ingestion.RAW_ROOT,
)
result = common_ingestion.ingest_nemweb_zip_files(sources, store, ingestion_config)
manifest_rows = result.manifest_rows
log_rows = result.log_rows


DispatchIS_Reports: 193 unseen ZIP(s)


## Review Ingestion Result

This cell previews the rows written by the selected store. Fabric displays a Spark DataFrame; local runs print the local CSV locations.


In [7]:
# Cell purpose: Review Ingestion Result.
if manifest_rows:
    if is_local_run:
        print(f"Manifest CSV: {store.manifest_path}")
        print(f"Ingestion log CSV: {store.log_path}")
        print(f"Rows written: {len(manifest_rows)}")
    else:
        display(spark.createDataFrame(manifest_rows))
else:
    print("No new ZIPs to ingest.")


Manifest CSV: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data\tables\raw_zip_manifest.csv
Ingestion log CSV: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data\tables\ingestion_log.csv
Rows written: 193
